**Data cleaning**
- ensure no duplicates records, all values in a certain column are the same data type (no mixed up types),
- look into if there are any missing data, for each of the 13 columns. 
- see if there are any anomalies

In [25]:
from zipfile import Path
import pandas as pd 
from pathlib import Path


In [26]:
df = pd.read_csv("/Users/sajnabhandari/ATLAS/im3_atlas.csv")
df
df.head()
df.tail() #ref & sqft -> 'NaN'

,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,lon,lat,type
1237,12387534800,New Mexico,NM,35,Bernalillo County,1,UNM,UNM,UNM Data Center,NaN,-106.615348,35.086739,point
1238,12437277166,California,CA,6,Santa Clara County,85,Prime,NaN,Prime Santa Clara,NaN,-121.962865,37.366662,point
1239,12452254071,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Open Data Centers Piscataway,NaN,-74.456856,40.550350,point
1240,12471740457,New Jersey,NJ,34,Hudson County,17,H5,NaN,H5 Secaucus,NaN,-74.075012,40.785395,point
1241,12550746951,California,CA,6,Santa Clara County,85,NaN,NaN,"Forescout Technologies, Inc.",NaN,-121.947418,37.322868,point


In [27]:
df.info()
#458 nulls in operator col
#915 nulls in ref col
#213 nulls in name
#93 nulls in sqft
df.duplicated().sum()
df.nunique()
df.describe
df.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 1242 entries, 0 to 1241
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         1242 non-null   int64  
 1   state      1242 non-null   str    
 2   state_abb  1242 non-null   str    
 3   state_id   1242 non-null   int64  
 4   county     1242 non-null   str    
 5   county_id  1242 non-null   int64  
 6   operator   784 non-null    str    
 7   ref        327 non-null    str    
 8   name       1029 non-null   str    
 9   sqft       1149 non-null   float64
 10  lon        1242 non-null   float64
 11  lat        1242 non-null   float64
 12  type       1242 non-null   str    
dtypes: float64(3), int64(3), str(7)
memory usage: 126.3 KB


id             0
state          0
state_abb      0
state_id       0
county         0
county_id      0
operator     458
ref          915
name         213
sqft          93
lon            0
lat            0
type           0
dtype: int64

In [ ]:
df = df.drop_duplicates() #drop any duplicates in row
df = df.drop_duplicates(subset=['id']).reset_index(drop=True) # dropping duplicates based on id column and resetting the index of the dataframe,
# so that the index is in a continuous sequence after dropping duplicates

df['operator'] = df['operator'].fillna('Unknown') # replacing null values in operator column with 'Unknown'
df['ref'] = df['ref'].fillna('Unknown') # replacing null values in ref column with 'Unknown'
df['name'] = df['name'].fillna('Unknown') # replacing null values in name column with 'Unknown'
median_sqft = df.loc[df['type'].isin(['building', 'campus']), 'sqft'].median() # calculating the median of sqft column for rows where type is either building or campus
#applying median only to the rows where type is either building or campus, as these are the only types that have sqft values
df.loc[df['type'].isin(['building', 'campus']) & (df['sqft'].isnull()), 'sqft'] 
#points will remain null, becuz they don't have sqft values, so we will not apply median to them
df['sqft'] = df['sqft'].fillna(0) # replacing null values in sqft column with 0, as points don't have sqft values

In [29]:
# making sure that the data types of the columns are correct, strings or ints
df['id'] = df['id'].astype(int)
df['state_id'] = df['state_id'].astype(int)
df['county_id'] = df['county_id'].astype(int)
#making sure that string columns as cleaned up
string_cols = ['state', 'state_abb', 'county', 'operator', 'ref', 'name', 'type']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()
# making sure that the numerical columns are of the correct data type-->floats
df['sqft'] = df['sqft'].astype(float)
df['lat'] = df['lat'].astype(float)
df['lon'] = df['lon'].astype(float)

In [ ]:
# checking for any negative values in sqft column and removing them
df = df[df['sqft'] > 0]
# checking for any negative values in lat and lon columns and removing them. 
# includes all US terriories (so puerto rico is included as well & not removed)
df = df[(df['lat'].between(17.0, 72.0)) & (df['lon'].between(-170.0, -65.0))]

In [31]:
print("Remaining Missing Values:/n", df.isnull().sum()) #re-checking for any missing values in the dataframe after cleaning
print("Remaining Duplicates:/n", df.duplicated().sum()) #re-checking for any duplicates in the dataframe after cleaning
df.info

Remaining Missing Values:/n id           0
state        0
state_abb    0
state_id     0
county       0
county_id    0
operator     0
ref          0
name         0
sqft         0
lon          0
lat          0
type         0
dtype: int64
Remaining Duplicates:/n 0


<bound method DataFrame.info of                id           state state_abb  state_id              county  \
0         2744301      New Jersey        NJ        34    Middlesex County   
1         7805491            Ohio        OH        39     Franklin County   
2         9474864  North Carolina        NC        37     Caldwell County   
3        13924557            Iowa        IA        19         Polk County   
4        14593270  North Carolina        NC        37      Catawba County   
...           ...             ...       ...       ...                 ...   
1237  12387534800      New Mexico        NM        35   Bernalillo County   
1238  12437277166      California        CA         6  Santa Clara County   
1239  12452254071      New Jersey        NJ        34    Middlesex County   
1240  12471740457      New Jersey        NJ        34       Hudson County   
1241  12550746951      California        CA         6  Santa Clara County   

      county_id   operator      ref        

In [34]:
df.to_csv("im3_atlas_cleaned.csv", index=False) # saving the cleaned dataframe to a new csv file